# PCA（主成分分析，1901）

- **核心原理**：最大變異方向投影
- **變體**：Kernel PCA（非線性）、Incremental PCA（資料放不進記憶體）、Randomized PCA（只要前幾個成分時很快）
- **推薦資料集**：Digits（1,797 張 8×8 手寫數字、64 個像素；第四章用過）

前面三章是「分群」：把**樣本**分組。PCA 是另一種非監督學習 —— **降維**：把**特徵**壓縮。64 個像素高度相關（相鄰的像素一起亮、一起暗），真正獨立變化的「方向」少很多（第四章 TODO 12：Digits 的 intrinsic dimension 約 8–20）。

PCA 找一組**新的座標軸**（主成分）：

1. 第一軸是資料**變異最大**的方向；
2. 第二軸是跟第一軸**垂直**的方向中，變異最大的；以此類推。

只保留前 $k$ 軸，就是把 64 維壓成 $k$ 維，而且是「保留最多變異」（等價地：**重建誤差最小**）的線性壓縮。

這一章的重點：**共變異數矩陣的特徵分解 = 主成分、SVD 是同一件事的另一種（更穩定的）算法、保留多少成分（解釋變異比例）、重建誤差 = 丟掉的特徵值、標準化要不要做、以及 PCA 拿來做什麼（視覺化、壓縮、去雜訊、加速下游模型、異常偵測）。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：每張影像 8×8 = 64 個像素（灰階 0–16），攤平成長度 64 的向量。欄名 `pixel_r_c` = 第 r 列、第 c 行（第四章）。標籤 0–9 只用於畫圖與下游分類。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$d$ | 樣本數、特徵數（像素） | 1,797、64 |
| $\bar x$ | 每個像素的平均（平均影像） | $(d,)$ |
| $X_c$ | 置中後的資料 $X - \bar x$ | $(n, d)$ |
| $C$ | 共變異數矩陣 $X_c^\top X_c / (n - 1)$ | $(d, d)$ |
| $\lambda_j$ | 第 $j$ 大的特徵值 = 第 $j$ 個主成分方向上的變異數 | $(d,)$ |
| $v_j$ | 對應的特徵向量（單位長度）= 第 $j$ 個主成分 | $(d,)$ |
| $V_k$ | 前 $k$ 個主成分排成的矩陣（每一**列**一個成分，同 sklearn 的 `components_`） | $(k, d)$ |
| $Z$ | 投影後的座標（分數，scores）$X_c V_k^\top$ | $(n, k)$ |
| $\hat X$ | 重建 $Z V_k + \bar x$ | $(n, d)$ |
| 解釋變異比例 | $\lambda_j / \sum_l \lambda_l$ | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第四章 | KNN（`01-classic-ml/04-knn`），同一份 Digits | 3 個像素恆為 0、12 個像素標準差 < 0.5（邊緣）；標準化會把這些雜訊像素放大（LOO 0.988 → 0.974）；PCA 2 維只保留約 28.5% 的變異、要 21 個成分才到 90%；維度詛咒：無關的維度讓距離失去意義。 |
| 第四章 TODO 1 (4) | 同上 | 正交投影只會讓距離變短；2 維的重疊不代表 64 維也重疊。 |
| 第三章 | 邏輯回歸 | 共線性：高度相關的特徵讓係數不穩定；條件數。 |
| 第十五章 | GMM | 共變異數矩陣 $\Sigma = \frac{1}{n}\sum (x - \mu)(x - \mu)^\top$；它的特徵向量就是 GMM 畫出來的橢圓的長短軸。 |
| 第 17 章 | t-SNE／UMAP | 非線性的降維，保留「鄰居關係」而不是「變異」；用 `trustworthiness` 比較。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 投影**：單位向量 $v$（$\lVert v\rVert = 1$），點 $x$ 在 $v$ 方向上的座標是 $x \cdot v$，投影點是 $(x \cdot v) v$。例：$x = (3, 4)$、$v = (1, 0)$ → 座標 3、投影點 $(3, 0)$。

**2. 投影後的變異數**：置中資料在方向 $v$ 上的座標 $z_i = x_i \cdot v$，變異數 $= \frac{1}{n-1}\sum_i z_i^2 = v^\top C v$。PCA 第一軸：在 $\lVert v\rVert = 1$ 的限制下最大化 $v^\top C v$ → 答案是 $C$ 最大特徵值的特徵向量，最大值就是那個特徵值（Rayleigh 商）。

**3. 對稱矩陣的特徵分解**：$C = V \Lambda V^\top$，$V$ 的欄是**互相垂直**的單位特徵向量、$\Lambda$ 是特徵值（實數、共變異數矩陣的都 $\ge 0$）。例：$C = \begin{pmatrix}2 & 1\\ 1 & 2\end{pmatrix}$ → 特徵值 3、1，特徵向量 $(1, 1)/\sqrt{2}$、$(1, -1)/\sqrt{2}$。特徵向量的**正負號不唯一**（$v$ 和 $-v$ 都是）。

**4. SVD**：任何矩陣 $X_c = U S V^\top$（$U$、$V$ 的欄正交、$S$ 是非負的奇異值）。代入 $C = X_c^\top X_c / (n - 1) = V \frac{S^2}{n - 1} V^\top$ → **$V$ 就是主成分、$\lambda_j = s_j^2 / (n - 1)$**。好處：不用先算 $X_c^\top X_c$（那會把條件數平方，損失精度）。

**5. 跡（trace）與總變異**：$\text{tr}(C) = \sum_j C_{jj} = \sum_j \lambda_j$ = 所有像素變異數的總和。所以「前 $k$ 個成分解釋的比例」$= \sum_{j \le k}\lambda_j / \text{tr}(C)$。

**6. 重建誤差**：只用前 $k$ 個成分重建，平方誤差總和 $\sum_i \lVert x_i - \hat x_i\rVert^2 = (n - 1)\sum_{j > k}\lambda_j$ —— 丟掉的特徵值加起來。（第 3C 節驗證。）

**7. 正交矩陣**：$V V^\top = I$ → 轉換到新座標再轉回來不會失真；只取前 $k$ 列 $V_k$ 時，$V_k^\top V_k$ 是「投影到前 $k$ 個方向」的矩陣。

#### 程式

**8. `np.linalg.eigh(C)`**：對稱矩陣專用；特徵值**由小到大**、特徵向量是**欄**。要由大到小：`order = np.argsort(vals)[::-1]`。浮點誤差可能讓 0 特徵值變成 −1e-15。
**9. `np.linalg.svd(Xc, full_matrices=False)`** → `U (n, r)`、`S (r,)`（由大到小）、`Vt (r, d)`（每一**列**是一個主成分），$r = \min(n, d)$。
**10. 對齊正負號**：比較兩組主成分時，`sign = np.sign((A * B).sum(axis=1))`，`A * sign[:, None]`。
**11. `np.cumsum`、`np.searchsorted(cum, 0.9)`**：累積比例第一次 $\ge 0.9$ 的位置。

---
## 1. 資料與探索性資料分析（EDA）

第四章已經看過這份資料；這裡只看跟 PCA 有關的：**像素之間的相關**與**變異集中在哪裡**。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_digits

X_df, y = load_digits(split=False)
X_df = pd.DataFrame(X_df)
X = X_df.to_numpy(dtype=float)
y = np.asarray(y).astype(int)
n, d = X.shape
print("X：", X.shape, "；每一類：", np.bincount(y).tolist())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(X.mean(axis=0).reshape(8, 8), cmap="gray_r")
axes[0].set_title("mean image")
axes[1].imshow(X.var(axis=0).reshape(8, 8), cmap="magma")
axes[1].set_title("pixel variance")
with np.errstate(invalid="ignore", divide="ignore"):  # 恆為 0 的像素：標準差 0，相關係數沒有定義
    corr = np.corrcoef(X.T)
corr = np.nan_to_num(corr)
axes[2].imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
axes[2].set_title("pixel-pixel correlation (64 × 64)")
plt.show()
iu = np.triu_indices(d, k=1)
print(f"兩兩像素相關係數 |r| > 0.5 的比例：{np.mean(np.abs(corr[iu]) > 0.5):.3f}；|r| > 0.8：{np.mean(np.abs(corr[iu]) > 0.8):.3f}")
print("變異數為 0 的像素：", [X_df.columns[j] for j in np.flatnonzero(X.var(axis=0) == 0)])

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 像素的變異集中在哪裡？（第四章 TODO 1 的重述）PCA 的第一個方向會比較「看」中央的像素還是邊緣的像素？為什麼？
# 答：
#
# (2) 相關矩陣：相鄰的像素（例如矩陣對角線旁邊）大多是正相關還是負相關？有沒有明顯的負相關？那代表什麼樣的筆畫？
#     「很多特徵高度相關」對 PCA 是好事還是壞事？（提示：相關 = 冗餘 = 可以壓縮）
# 答：
#
# (3) 有 3 個像素變異數是 0。它們在 PCA 裡會變成什麼？（提示：共變異數矩陣的秩 → 有幾個特徵值是 0？）
# 答：

---
## 2. Baseline：用其他方法挑 $k$ 個維度

把 64 維壓成 $k$ 維、再盡量重建回來，比較重建誤差（每個像素的平均平方誤差，MSE）：

- **挑變異最大的 $k$ 個像素**：保留這 $k$ 個像素的原值，其他像素用平均值補。
- **隨機投影**：隨機挑 $k$ 個（正交化過的）方向投影，再用同樣的方向重建。
- PCA 在第 3 節。

#### 本節名詞

- `reconstruction_error(X, X_hat)`（`learning_map.metrics`）：回傳 mse、rmse。

In [ ]:
from learning_map.metrics import reconstruction_error

mean_img = X.mean(axis=0)
Xc = X - mean_img
rng = np.random.default_rng(0)


def recon_top_variance_pixels(X: np.ndarray, k: int) -> np.ndarray:
    """保留變異最大的 k 個像素，其他補平均。"""
    keep = np.argsort(X.var(axis=0))[::-1][:k]
    X_hat = np.tile(X.mean(axis=0), (len(X), 1))
    X_hat[:, keep] = X[:, keep]
    return X_hat


def recon_random_projection(X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
    """隨機的 k 個正交方向（QR 分解把隨機矩陣正交化）。"""
    Q, _ = np.linalg.qr(rng.normal(size=(X.shape[1], k)))  # (d, k)，欄互相垂直、長度 1
    mu = X.mean(axis=0)
    return (X - mu) @ Q @ Q.T + mu


ks = [2, 5, 10, 20, 30, 40]
baseline = pd.DataFrame(
    {
        "top-variance pixels": [reconstruction_error(X, recon_top_variance_pixels(X, k))["mse"] for k in ks],
        "random projection": [reconstruction_error(X, recon_random_projection(X, k, rng))["mse"] for k in ks],
    },
    index=pd.Index(ks, name="k"),
)
print("k = 0（全部用平均影像）的 MSE：", round(reconstruction_error(X, np.tile(mean_img, (n, 1)))["mse"], 3), "= 每個像素變異數的平均")
display(baseline.round(3))

In [ ]:
# TODO 2（問答）
#
# (1) k = 0 的 MSE 為什麼剛好等於「每個像素變異數的平均」？
# 答：
#
# (2) 「保留變異最大的 k 個像素」和「隨機投影」在 k 小、k 大時各誰比較好？
#     為什麼挑像素有它的極限？（提示：像素高度相關 —— 兩個幾乎一樣的像素，挑了一個之後再挑另一個，多得到多少資訊？）
#     PCA 的方向可以是「很多像素的組合」，這對相關的資料有什麼好處？
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 共變異數 → 特徵分解 → 主成分；投影與重建 | 3、4、5 |
| 3B | 用 SVD 算（sklearn 的做法） | 6 |
| 3C | 實驗：解釋變異、重建影像與誤差、eigen-digits、標準化、PCA + KNN | — |

### 3A. 特徵分解

1. **置中**：$X_c = X - \bar x$。（不置中的話，第一個「方向」會指向平均影像，而不是變異最大的方向。）
2. **共變異數**：$C = X_c^\top X_c / (n - 1)$（TODO 3）。
3. **特徵分解**：`eigh`，由大到小排序（TODO 4）。前 $k$ 個特徵向量（轉置成列）就是 `components_`。
4. **投影**：$Z = X_c V_k^\top$；**重建**：$\hat X = Z V_k + \bar x$（TODO 5）。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $\bar x$ | 平均 | $(d,)$ | `self.mean_` |
| $C$ | 共變異數 | $(d, d)$ | `C` |
| vals、vecs | `eigh` 的輸出（由小到大；vecs 的**欄**是特徵向量） | $(d,)$、$(d, d)$ | `vals`、`vecs` |
| $\lambda$ | 由大到小的特徵值 | $(d,)$ | `self.explained_variance_`（只留前 $k$ 個） |
| $V_k$ | 前 $k$ 個主成分（**列**） | $(k, d)$ | `self.components_` |
| $Z$ | 分數 | $(n, k)$ | `transform` 的回傳 |

**下標公式**：
- TODO 3：$C_{jl} = \frac{1}{n-1}\sum_i X_{c,ij} X_{c,il}$ —— 加總 $i$ → `Xc.T @ Xc / (n - 1)`。
- TODO 5：$Z_{ij} = \sum_l X_{c,il} V_{jl}$ → `Xc @ V.T`；$\hat X_{il} = \sum_j Z_{ij} V_{jl} + \bar x_l$ → `Z @ V + mean`。

#### 最小例子：TODO 3、4、5（二維的 PCA）

4 個點：$(1, 1), (2, 2), (3, 3), (4, 5)$ —— 幾乎在一條斜線上。

1. **正確答案**：手算／迴圈。$\bar x = (2.5, 2.75)$；$C = \begin{pmatrix}1.667 & 2.167\\ 2.167 & 2.917\end{pmatrix}$；第一個主成分約 $(0.601, 0.799)$（斜線方向），特徵值約 4.547，佔 99.2%；第二個特徵值約 0.037。
2. **積木**：`Xc.T @ Xc`、`eigh` 的輸出順序、`[::-1]`、`vecs[:, order]`。
3. **留給你**：寫進 `fit`、`transform`、`inverse_transform`。

In [ ]:
# 最小例子：TODO 3、4、5（二維的 PCA）
X_mini = np.array([[1.0, 1.0], [2.0, 2.0], [3.0, 3.0], [4.0, 5.0]])

# ---- 1. 正確答案：迴圈算共變異數，numpy 算特徵 ----
mu_loop = X_mini.mean(axis=0)
C_loop = np.zeros((2, 2))
for x in X_mini:
    C_loop += np.outer(x - mu_loop, x - mu_loop)
C_loop /= len(X_mini) - 1
print("平均", mu_loop, "\nC =\n", C_loop.round(4), "\n（np.cov 對照：", np.allclose(C_loop, np.cov(X_mini.T)), "）")

# ---- 2. 積木 ----
vals_mini, vecs_mini = np.linalg.eigh(C_loop)
print("\neigh：vals =", vals_mini.round(4), "← 由小到大；vecs 的「欄」是特徵向量：\n", vecs_mini.round(4))
order_mini = np.argsort(vals_mini)[::-1]
print("由大到小：vals[order] =", vals_mini[order_mini].round(4), "；第一個主成分 = vecs[:, order[0]] =", vecs_mini[:, order_mini[0]].round(4))
print("解釋比例：", (vals_mini[order_mini] / vals_mini.sum()).round(4))

# ---- 3. 留給你 ----
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（components_ 可能差一個負號）：
# p_mini = PCAScratch(1).fit(X_mini); print(p_mini.components_.round(4), p_mini.explained_variance_.round(4), p_mini.inverse_transform(p_mini.transform(X_mini)).round(3))

In [ ]:
from typing import Literal, Self

from numpy.typing import ArrayLike


class PCAScratch:
    """PCA。method："eigh"（共變異數的特徵分解）或 "svd"（置中資料的 SVD）。屬性名稱同 sklearn。"""

    def __init__(self, n_components: int | None = None, method: Literal["eigh", "svd"] = "eigh") -> None:
        self.n_components = n_components
        self.method = method

    def fit(self, X: ArrayLike) -> Self:
        X = np.asarray(X, dtype=float)
        n, d = X.shape
        k = d if self.n_components is None else self.n_components
        self.mean_ = X.mean(axis=0)
        Xc = X - self.mean_
        if self.method == "eigh":
            # TODO 3（程式）：共變異數矩陣 C (d, d)
            raise NotImplementedError("TODO 3：共變異數")
            vals, vecs = np.linalg.eigh(C)
            # TODO 4（程式）：由大到小排序
            #   - order = np.argsort(vals)[::-1]；all_var = vals[order]；comps = vecs[:, order].T（每一列一個成分）
            raise NotImplementedError("TODO 4：排序")
        else:
            all_var, comps = self._svd(Xc)
        self.components_ = comps[:k]
        self.explained_variance_ = all_var[:k]
        self.explained_variance_ratio_ = all_var[:k] / all_var.sum()
        self.all_explained_variance_ = all_var  # 全部的特徵值（實驗用）
        return self

    @staticmethod
    def _svd(Xc: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """回傳 (全部的變異數 λ（由大到小）, 全部的主成分 (r, d))。"""
        n = len(Xc)
        # TODO 6（程式）：用 SVD
        #   - U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
        #   - λ = S² / (n − 1)；主成分 = Vt（已經由大到小）
        raise NotImplementedError("TODO 6：SVD")

    def transform(self, X: ArrayLike) -> np.ndarray:
        """(n, k)：投影座標。"""
        # TODO 5（程式）之一：Z = (X − mean_) @ components_.T
        raise NotImplementedError("TODO 5：transform")

    def inverse_transform(self, Z: ArrayLike) -> np.ndarray:
        """(n, d)：從 k 維座標重建。"""
        # TODO 5（程式）之二：X̂ = Z @ components_ + mean_
        raise NotImplementedError("TODO 5：inverse_transform")

### 3B. 用 SVD 算 PCA

前置複習 4：$X_c = U S V^\top$ → $C = V \frac{S^2}{n-1} V^\top$。所以：

- 主成分 = $V^\top$ 的列（`Vt`），已經由大到小；
- $\lambda_j = s_j^2 / (n - 1)$；
- 分數 $Z = X_c V = U S$（不用再乘一次）。

為什麼 sklearn 用 SVD 不用 `eigh`？算 $X_c^\top X_c$ 會把條件數**平方**：小的特徵值（$10^{-15}$ 那種）會先被浮點誤差吃掉，甚至變成負的。SVD 直接分解 $X_c$，沒有這一步。

**正負號**：特徵向量 $v$ 與 $-v$ 都對（前置複習 3）。sklearn 用 `svd_flip` 固定一個慣例；比較時要先對齊（前置複習 10）。

（TODO 6 寫在上一格的 `_svd` 裡。）

In [ ]:
from sklearn.decomposition import PCA


def align_signs(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """把 A 的每一列乘上 ±1，讓它跟 B 的同一列同向。"""
    s = np.sign((A * B).sum(axis=1))
    s[s == 0] = 1
    return A * s[:, None]


sk = PCA().fit(X)
for method in ["eigh", "svd"]:
    mine = PCAScratch(method=method).fit(X)
    print(f"[{method}] ✅ 變異數最大差異 {np.abs(mine.explained_variance_ - sk.explained_variance_).max():.1e}（應 < 1e-10）",
          f"；前 40 個主成分（對齊正負號後）最大差異 {np.abs(align_signs(mine.components_[:40], sk.components_[:40]) - sk.components_[:40]).max():.1e}（應 < 1e-8）")
    print("      最小的 5 個 λ：", mine.all_explained_variance_[-5:])
p10 = PCAScratch(10, method="svd").fit(X)
Z_mine = p10.transform(X)
Z_sk = PCA(10).fit_transform(X)
print("✅ transform（對齊正負號後）最大差異：", np.abs(align_signs(Z_mine.T, Z_sk.T).T - Z_sk).max(), "（應 < 1e-8）")
print("✅ 主成分互相垂直、長度 1：V Vᵀ = I？", np.allclose(p10.components_ @ p10.components_.T, np.eye(10)))
print("✅ 分數之間不相關（Zᵀ Z / (n−1) 是對角矩陣 = λ）：", np.allclose(Z_mine.T @ Z_mine / (n - 1), np.diag(p10.explained_variance_)))

In [ ]:
# TODO 7（問答）
#
# (1) 最小例子：第一個主成分是什麼方向？它解釋了多少變異？只用 1 維重建，4 個點會落在哪裡？哪一個點的重建誤差最大？
# 答：
#
# (2) 為什麼 PCA 一定要先置中？如果不置中，直接對 XᵀX 做特徵分解，第一個「主成分」會指向哪裡？
# 答：
#
# (3) eigh 與 svd：
#     (a) 最小的幾個 λ 各是多少？有沒有負的？λ 理論上能是負的嗎？為什麼會出現？
#     (b) 有幾個 λ ≈ 0？跟 TODO 1 (3) 的預測一樣嗎？
#     (c) 「對齊正負號後」才一樣 —— 為什麼主成分的正負號沒有意義？分數 Z 的正負號呢？
# 答：
#
# (4) 「分數之間不相關」的檢查通過了：PCA 把 64 個「互相相關」的像素，換成 k 個「互相不相關」的新特徵。
#     對第三章的邏輯回歸（共線性讓係數不穩）來說，這有什麼用？
# 答：

### 3C. 實驗：解釋變異、重建、eigen-digits、標準化、PCA + KNN

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| scree plot | 特徵值（或解釋比例）依大小排的圖 |
| eigen-digits | 把主成分（長度 64 的向量）排成 8×8 的影像：每個主成分是一種「筆畫的變化模式」 |
| 相關矩陣的 PCA | 先標準化再做 PCA（= 對相關矩陣做特徵分解） |

In [ ]:
# 實驗 1：解釋變異比例與重建誤差
pfull = PCAScratch(method="svd").fit(X)
ratio = pfull.all_explained_variance_ / pfull.all_explained_variance_.sum()
cum = np.cumsum(ratio)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(np.arange(1, 21), ratio[:20])
axes[0].set_xlabel("component")
axes[0].set_ylabel("explained variance ratio")
axes[0].set_title("Scree plot (first 20)")
axes[1].plot(np.arange(1, d + 1), cum, "o-", ms=3)
for thr in [0.8, 0.9, 0.95]:
    axes[1].axhline(thr, color="gray", ls=":", lw=1)
axes[1].set_xlabel("number of components k")
axes[1].set_ylabel("cumulative explained variance")
plt.show()
print({f"{int(thr * 100)}%": int(np.argmax(cum >= thr)) + 1 for thr in [0.5, 0.8, 0.9, 0.95, 0.99]})

rows = []
for k in [1, 2, 5, 10, 20, 30, 40]:
    p = PCAScratch(k, method="svd").fit(X)
    X_hat = p.inverse_transform(p.transform(X))
    rows.append({"k": k, "PCA MSE": reconstruction_error(X, X_hat)["mse"], "Σ discarded λ · (n−1)/(n·d)": pfull.all_explained_variance_[k:].sum() * (n - 1) / (n * d)})
rec_tbl = pd.DataFrame(rows).set_index("k").join(baseline)
display(rec_tbl.round(4))

In [ ]:
# 實驗 2：重建影像與 eigen-digits
fig, axes = plt.subplots(6, 8, figsize=(11, 8.5))
idx_show = [np.flatnonzero(y == c)[0] for c in [0, 2, 3, 4, 7, 8, 9, 1]]
for c, i in enumerate(idx_show):
    axes[0, c].imshow(X[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    for r, k in enumerate([2, 5, 10, 21, 40], start=1):
        p = PCAScratch(k, method="svd").fit(X)
        axes[r, c].imshow(p.inverse_transform(p.transform(X[i : i + 1])).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
for r, lab in enumerate(["original", "k=2", "k=5", "k=10", "k=21", "k=40"]):
    axes[r, 0].set_ylabel(lab)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("Reconstructions with k components")
plt.show()

fig, axes = plt.subplots(1, 9, figsize=(16, 2.2))
axes[0].imshow(pfull.mean_.reshape(8, 8), cmap="gray_r")
axes[0].set_title("mean")
for j in range(8):
    v = pfull.components_[j].reshape(8, 8)
    lim = np.abs(v).max()
    axes[j + 1].imshow(v, cmap="coolwarm", vmin=-lim, vmax=lim)
    axes[j + 1].set_title(f"PC{j + 1} ({ratio[j]:.1%})")
for ax in axes:
    ax.axis("off")
plt.show()

In [ ]:
# 實驗 3：標準化（相關矩陣的 PCA）vs 不標準化；PCA 2D 圖
from sklearn.preprocessing import StandardScaler

Xs = StandardScaler().fit_transform(X)
ps = PCAScratch(method="svd").fit(Xs)
cum_s = np.cumsum(ps.all_explained_variance_ / ps.all_explained_variance_.sum())
print("達到 90% 需要的成分數：raw", int(np.argmax(cum >= 0.9)) + 1, "；standardized", int(np.argmax(cum_s >= 0.9)) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, (name, p, Xin) in zip(axes[:2], [("raw pixels", pfull, X), ("standardized pixels", ps, Xs)]):
    Z = p.transform(Xin)[:, :2]
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=6)
    for c in range(10):
        ax.text(*np.median(Z[y == c], axis=0), str(c), fontsize=14, weight="bold")
    ax.set_title(f"PCA 2D on {name}")
v = ps.components_[0].reshape(8, 8)
lim = np.abs(v).max()
axes[2].imshow(v, cmap="coolwarm", vmin=-lim, vmax=lim)
axes[2].set_title("PC1 of standardized pixels")
plt.show()

In [ ]:
# 實驗 4：PCA + KNN（第四章）—— 降到 k 維再做 KNN，5-fold CV accuracy 與速度
import time

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(5, shuffle=True, random_state=42)
rows = []
for k in [2, 5, 10, 15, 20, 30, 40, 64]:
    model = KNeighborsClassifier(3) if k == 64 else make_pipeline(PCA(k), KNeighborsClassifier(3))
    t0 = time.perf_counter()
    s = cross_val_score(model, X, y, cv=cv)
    rows.append({"k": k, "CV accuracy": s.mean(), "std": s.std(), "seconds": time.perf_counter() - t0})
knn_tbl = pd.DataFrame(rows).set_index("k")
display(knn_tbl.round(4))

In [ ]:
# TODO 8（問答）實驗 1–4
#
# (1) 實驗 1：
#     (a) 要幾個成分才到 80%、90%、95%？跟第四章說的「21 個到 90%」一樣嗎？
#     (b) 表格的「PCA MSE」和「Σ 丟掉的 λ」兩欄一樣嗎？用前置複習 6 解釋。
#     (c) 同樣的 k，PCA 的 MSE 比「挑像素」「隨機投影」小多少？PCA 是「線性壓縮裡重建誤差最小」的，你能從 (b) 說出為什麼嗎？
# 答：
#
# (2) 實驗 2：
#     (a) k = 2、5 的重建影像看得出是什麼數字嗎？k 多少開始看得出來？
#     (b) PC1、PC2 的影像：紅色（正）和藍色（負）的區域各在哪裡？PC1 的分數大，代表影像長什麼樣子？（試著跟 0 或 1 連結）
# 答：
#
# (3) 實驗 3：
#     (a) 標準化之後，要多少個成分才到 90%？比 raw 多還是少？為什麼？（提示：標準化後每個像素的變異都是 1，連邊緣那些「幾乎不變」的像素也是）
#     (b) PC1 of standardized pixels 的圖：邊緣像素的權重變大了嗎？（第四章 TODO 11：標準化把雜訊像素放大）
#     (c) 結論：像素資料做 PCA 要不要標準化？什麼樣的資料「一定要」標準化？（提示：單位不同的特徵 —— 第十三章的 Wholesale）
# 答：
#
# (4) 實驗 4：
#     (a) k 多少時 KNN 的 accuracy 已經跟 64 維差不多？k 很小（2、5）時差多少？
#     (b) 有沒有某個 k 比 64 維還好？如果有，PCA 丟掉了什麼？（提示：最後那些小 λ 的方向大多是雜訊；對照第四章維度詛咒）
#     (c) 速度有變快嗎？在這麼小的資料上，降維的速度好處明顯嗎？
# 答：

---
## 4. 套件實作：`PCA`、`IncrementalPCA`、`KernelPCA`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `PCA(n_components=...)` | 整數 = 成分數；0–1 的小數 = 「保留這個比例的變異」所需的最少成分數；`"mle"` = Minka 的自動選擇 |
| `svd_solver` | `"full"`（完整 SVD）、`"randomized"`（隨機演算法，只要前幾個成分時很快）、`"auto"` |
| `whiten=True` | 分數再除以 $\sqrt{\lambda}$，讓每個成分的變異數都是 1 |
| `IncrementalPCA` | 一批一批讀資料更新（`partial_fit`），記憶體只需要一批 |
| `KernelPCA` | 把資料用 kernel（第八章）映射到高維，再在那裡做 PCA → 非線性的降維 |

In [ ]:
from sklearn.datasets import make_circles
from sklearn.decomposition import IncrementalPCA, KernelPCA

# 速度：大一點的隨機矩陣，只要前 10 個成分
X_big = np.random.default_rng(0).normal(size=(20000, 500)) @ np.random.default_rng(1).normal(size=(500, 500))
rows = []
for name, make in {"PCA full": lambda: PCA(10, svd_solver="full"), "PCA randomized": lambda: PCA(10, svd_solver="randomized", random_state=0), "IncrementalPCA (batch 2000)": lambda: IncrementalPCA(10, batch_size=2000)}.items():
    t0 = time.perf_counter()
    m = make().fit(X_big)
    rows.append({"method": name, "seconds": time.perf_counter() - t0, "explained var of PC1 (relative to full)": m.explained_variance_[0]})
speed = pd.DataFrame(rows).set_index("method")
speed.iloc[:, 1] /= speed.iloc[0, 1]
display(speed.round(4))

p90 = PCA(0.9).fit(X)
print("PCA(0.9) 選了", p90.n_components_, "個成分")

# Kernel PCA：同心圓
Xc2, yc2 = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=0)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(Xc2[:, 0], Xc2[:, 1], c=yc2, cmap="coolwarm", s=8)
axes[0].set_title("make_circles")
Zl = PCA(2).fit_transform(Xc2)
axes[1].scatter(Zl[:, 0], np.zeros_like(Zl[:, 0]) + 0.02 * yc2, c=yc2, cmap="coolwarm", s=8)
axes[1].set_title("linear PCA: PC1 only")
Zk = KernelPCA(2, kernel="rbf", gamma=10).fit_transform(Xc2)
axes[2].scatter(Zk[:, 0], np.zeros_like(Zk[:, 0]) + 0.02 * yc2, c=yc2, cmap="coolwarm", s=8)
axes[2].set_title("Kernel PCA (RBF, gamma=10): PC1 only")
plt.show()

In [ ]:
# TODO 9（問答）
#
# (1) randomized SVD 比 full 快多少？PC1 的變異數差多少？什麼時候值得用？
# 答：
#
# (2) Kernel PCA：線性 PCA 的 PC1 能把內圈和外圈分開嗎？Kernel PCA 呢？
#     用第八章的 kernel trick 解釋：RBF kernel 對應的特徵空間裡，「離中心的距離」變成了什麼？
# 答：

---
## 5. 選成分數

| 方法 | 做法 |
| --- | --- |
| 解釋變異的門檻 | 最少要幾個成分，累積比例才 $\ge$ 門檻（80%／90%／95%） —— TODO 10 |
| scree plot 的轉折 | 跟第十四章的 k-distance 一樣找「轉折點」 |
| 下游任務的 CV | 把 PCA 放進 Pipeline，用 GridSearchCV 選 `n_components`（例如實驗 4） |
| 重建誤差的門檻 | 例如每個像素的 RMSE < 1 |

#### 最小例子：TODO 10（門檻 → 成分數）

解釋比例 `[0.5, 0.2, 0.15, 0.1, 0.05]`，門檻 0.8。

1. **正確答案**：累積 `[0.5, 0.7, 0.85, 0.95, 1.0]`，第一次 ≥ 0.8 是第 3 個 → 3。
2. **積木**：`np.cumsum`、`np.searchsorted(cum, thr)`（回傳「插入位置」，從 0 開始）。
3. **留給你**：處理浮點誤差（累積到最後可能是 0.9999999999），並把 0 起算的位置換成「個數」。

In [ ]:
# 最小例子：TODO 10（門檻 → 成分數）
r_mini = np.array([0.5, 0.2, 0.15, 0.1, 0.05])
cum_mini = np.cumsum(r_mini)
print("累積：", cum_mini, "；迴圈答案：", next(i + 1 for i, c in enumerate(cum_mini) if c >= 0.8))
print("np.searchsorted(cum, 0.8) =", np.searchsorted(cum_mini, 0.8), "← 0 起算的位置")
print("浮點誤差：np.cumsum([0.1] * 10)[-1] =", repr(np.cumsum([0.1] * 10)[-1]), "← 門檻設 1.0 會找不到")
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照（應為 3、5）：
# print(n_components_for(r_mini, 0.8), n_components_for(r_mini, 1.0))

In [ ]:
def n_components_for(ratio: np.ndarray, threshold: float) -> int:
    """解釋比例（由大到小）→ 累積比例 ≥ threshold 的最少成分數。"""
    cum = np.cumsum(ratio)
    # TODO 10（程式）
    #   - 用 np.searchsorted(cum, threshold − 1e-12) 找位置（減一點點，容忍浮點誤差）
    #   - 位置是 0 起算 → 成分數 = 位置 + 1；最多 len(ratio)
    raise NotImplementedError("TODO 10：門檻 → 成分數")


for thr in [0.8, 0.9, 0.95, 0.99]:
    print(f"✅ 門檻 {thr}：手寫 {n_components_for(ratio, thr)}、sklearn PCA({thr}).n_components_ = {PCA(thr).fit(X).n_components_}")

from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(make_pipeline(PCA(), KNeighborsClassifier(3)), {"pca__n_components": [5, 10, 15, 20, 25, 30, 40, 50]}, cv=cv, n_jobs=-1).fit(X, y)
gs_res = pd.DataFrame({"n_components": gs.cv_results_["param_pca__n_components"].astype(int), "CV acc": gs.cv_results_["mean_test_score"], "std": gs.cv_results_["std_test_score"]})
display(gs_res.round(4).T)
print("下游 KNN 的 CV 最佳 n_components：", gs.best_params_)

In [ ]:
# TODO 11（問答）選成分數
#
# (1) 門檻 80%、90%、95%、99% 各要幾個成分？從 95% 到 99% 要多付多少個成分？
# 答：
#
# (2) 下游 KNN 的 CV 最佳 n_components 是多少？它對應多少 % 的變異？
#     「保留 90% 變異」和「下游分類最好」選出來的 k 一樣嗎？如果不一樣，你會用哪一個？取決於什麼？
# 答：

---
## 6. 評估

降維沒有「正確答案」，用兩個角度：

- **重建誤差**（`reconstruction_error`）：保留多少資訊。
- **trustworthiness**（`trustworthiness(X_high, X_low, n_neighbors=5)`，0–1）：低維中的鄰居，在高維中也是鄰居的程度（第 17 章 t-SNE／UMAP 會用同一個指標比較）。

成績寫進 `results/digits_embedding.csv`（第 17 章會接著加 t-SNE、UMAP）。

In [ ]:
from learning_map.metrics import trustworthiness
from learning_map.results import compare, log_result

BENCH = "digits_embedding"
runs = {}
for k in [2, 10, 21]:
    p = PCAScratch(k, method="svd").fit(X)
    Z = p.transform(X)
    runs[f"PCA k={k} (scratch)"] = ({**reconstruction_error(X, p.inverse_transform(Z)), **trustworthiness(X, Z), "explained_var": round(float(p.explained_variance_ratio_.sum()), 4)}, {"k": k})
Zs = PCAScratch(2, method="svd").fit(Xs).transform(Xs)
runs["PCA k=2 standardized (scratch)"] = ({**trustworthiness(X, Zs)}, {"k": 2, "standardized": True})
Q, _ = np.linalg.qr(np.random.default_rng(0).normal(size=(d, 2)))
runs["Random projection k=2"] = ({**reconstruction_error(X, recon_random_projection(X, 2, np.random.default_rng(0))), **trustworthiness(X, Xc @ Q)}, {"k": 2})
scores = {}
for name, (m, params) in runs.items():
    scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(scores).T)
compare(BENCH, "trustworthiness")

---
## 7. 診斷：重建不好的影像 = 不典型的影像

用 $k = 21$（90% 變異）重建；重建誤差特別大的影像，就是「主要的變化模式解釋不了」的影像 —— 可以拿來找異常或標錯的樣本。

In [ ]:
p21 = PCAScratch(21, method="svd").fit(X)
err = ((X - p21.inverse_transform(p21.transform(X))) ** 2).mean(axis=1)
worst = np.argsort(err)[::-1][:8]
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for c, i in enumerate(worst):
    axes[0, c].imshow(X[i].reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
    axes[0, c].set_title(f"label {y[i]}\nMSE {err[i]:.1f}", fontsize=9)
    axes[1, c].imshow(p21.inverse_transform(p21.transform(X[i : i + 1])).reshape(8, 8), cmap="gray_r", vmin=0, vmax=16)
axes[0, 0].set_ylabel("original")
axes[1, 0].set_ylabel("k=21")
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()
per_class = pd.Series(err).groupby(y).mean().round(2)
print("各數字的平均重建誤差：", per_class.to_dict())
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(err, bins=50)
ax.set_xlabel("reconstruction MSE per image (k = 21)")
plt.show()

In [ ]:
# TODO 12（問答，本章總結）
#
# (1) 重建誤差最大的影像：
#     (a) 它們寫得有什麼特別？是寫法罕見、還是可能標錯？
#     (b) 哪個數字的平均重建誤差最大？為什麼那個數字特別難用 21 個主成分表示？
#     (c) 「重建誤差大 = 異常」可以當異常偵測嗎？跟第十五章「GMM 的 log 密度低 = 異常」相比，兩者各在看什麼？
# 答：
#
# (2) 第一部分非監督學習四章的比較表（填空）：
#
#     |              | K-Means | DBSCAN | GMM | PCA |
#     | 處理的對象   | 樣本／特徵 |     |     |     |
#     | 輸出         |         |        |     |     |
#     | 主要假設     |         |        |     |     |
#     | 要選的超參數 |         |        |     |     |
#     | 演算法       |         |        |     |     |
#
# (3) PCA 是線性的：只能沿著「直線方向」壓縮。第 17 章 t-SNE／UMAP 是非線性的，它們放棄了什麼、換到了什麼？（先猜）
#     PCA 2D 圖上 1、7、8、5、2 重疊（第四章 EDA），你預期 t-SNE 會把它們分開嗎？
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Digits（1,797 張、64 個像素；標籤只用於畫圖與下游分類）。

| 方法 | 做法 | 關鍵性質 | 重建 MSE（k = 10） | trustworthiness（k = 2） |
| --- | --- | --- | --- | --- |
| 挑變異最大的像素 | | | | — |
| 隨機投影 | | | | |
| PCA（手寫，eigh） | 共變異數的特徵分解 | | | |
| PCA（手寫，SVD） | 置中資料的 SVD | | | |
| PCA（標準化後） | | | — | |
| PCA + KNN | 下游分類 | | — | — |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ PCA 的兩種推導：最大變異（Rayleigh 商、拉格朗日乘數）與最小重建誤差（Eckart–Young 定理）
- ☐ 機率 PCA（PPCA）與因子分析（Factor Analysis）；PCA 與 GMM 的關係
- ☐ 隨機 SVD（Halko, Martinsson & Tropp 2011）
- ☐ 主成分的統計推論：要保留幾個成分的檢定（parallel analysis、Minka 的 MLE）

**延伸**
- ☐ Whitening、ZCA；PCA 去雜訊（只用前 k 個成分重建）
- ☐ Kernel PCA 的 pre-image 問題；其他非線性降維（Isomap、LLE）
- ☐ 稀疏 PCA、NMF（非負矩陣分解）、ICA（獨立成分分析）
- ☐ 推薦系統的矩陣分解（SVD 的應用）；LSA（文字的 SVD）
- ☐ 第 17 章 t-SNE／UMAP；自編碼器（第四部分的 Autoencoder）是 PCA 的非線性版